# Standalone CONUS404 runoff aggregation with NLDI basins and USGS observations

This executable notebook preserves the complete runoff workflow from `Sanford_Basins_Runoff-Annual.ipynb` without requiring `config.yml`, prepared runoff tables, basin shapefiles, or gage-point files as inputs. All scientific inputs are retrieved from public web services:

- the HyTEST intake catalog supplies daily CONUS404 runoff components;
- NLDI supplies USGS-gage contributing-basin polygons and gage locations;
- NWIS supplies daily mean discharge observations.

The workflow forms annual water-year CONUS404 runoff, constructs grid-cell polygons, intersects them with NLDI basins, calculates area fractions, builds a sparse basin-weight matrix, aggregates gridded runoff to basin means, converts USGS discharge to runoff depth, aligns the annual records, calculates accuracy statistics, and creates the time-series, regression, and basin-area figures used to inspect model behavior.

The complete CAMELS gage list from the source notebook is embedded below. The default execution processes the first eight gages so the workflow can be demonstrated without launching more than one thousand remote basin and streamflow requests. Setting `RUN_FULL_GAGE_SET = True` applies the identical operations to the complete embedded list.


## 1. Runtime parameters and libraries

The parameter cell contains every user-adjustable execution value. No machine-specific input path is required. The output directory is created inside the repository.


In [ ]:
from pathlib import Path
import math
import time

import cf_xarray
import dask
import geopandas as gpd
import hydrofunctions as hf
import intake
import matplotlib.pyplot as plt
import metpy
import numpy as np
import pandas as pd
import rioxarray
import sparse
import xarray as xr
from dask.distributed import Client
from pynhd import NLDI
from shapely.geometry import Polygon
from sklearn.linear_model import LinearRegression

REPOSITORY_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
OUTPUT_DIRECTORY = REPOSITORY_ROOT / "results" / "runoff_hytest_nldi"
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
SCATTER_DIRECTORY = OUTPUT_DIRECTORY / "scatter_plots"
SCATTER_DIRECTORY.mkdir(parents=True, exist_ok=True)

HYTEST_CATALOG_URL = "https://raw.githubusercontent.com/hytest-org/hytest/main/dataset_catalog/hytest_intake_catalog.yml"
CONUS404_DATASET = "conus404-daily-osn"
START_DATE = "1989-10-01"
END_DATE = "2010-09-30"
RUN_FULL_GAGE_SET = False
DEMONSTRATION_GAGE_COUNT = 8
MINIMUM_DAILY_COVERAGE = 0.90
MINIMUM_REGRESSION_PAIRS = 3
SCATTER_GRID_R2_MINIMUM = 0.70
SCATTER_GRID_PAIR_MINIMUM = 15
DASK_WORKERS = 4


## 2. Embedded CAMELS gage set

The gage identifiers are embedded from the attached source notebook. This removes the source workflow's dependency on `CONUS404_gages_list.txt` while retaining the complete basin selection.


In [ ]:
ALL_CAMELS_GAGE_IDS = ['01013500', '01019000', '01022500', '01030500', '01031500', '01038000', '01047000', '01052500', '01054200', '01055000', '01057000', '01064500', '01073000', '01073500', '01078000', '01094500', '01096000', '01097000', '01102000', '01104500', '01109000', '01109060', '01111500', '01116000', '01116500', '01118300', '01118500', '01119500', '01121000', '01123000', '01127500', '01129200', '01134500', '01137500', '01139000', '01139800', '01142500', '01144000', '01151500', '01152500', '01162500', '01166500', '01168500', '01169000', '01170100', '01171500', '01174500', '01176000', '01181000', '01185500', '01186000', '01186500', '01187300', '01193500', '01195100', '01196500', '01197000', '01197500', '01204000', '01208500', '01315000', '01331500', '01333000', '01334000', '01336000', '01350000', '01350080', '01350140', '01365000', '01367500', '01372500', '01380500', '01388000', '01388500', '01391500', '01395000', '01396800', '01397000', '01400000', '01401000', '01402000', '01405400', '01408000', '01408500', '01409400', '01410000', '01411000', '01411300', '01411500', '01413500', '01414500', '01415000', '01420500', '01423000', '01429000', '01429500', '01431500', '01434025', '01435000', '01439500', '01440000', '01440400', '01442500', '01445500', '01447500', '01447720', '01449360', '01450500', '01451500', '01451800', '01452000', '01452500', '01457000', '01464000', '01464500', '01465500', '01466500', '01467000', '01467048', '01469500', '01470500', '01472157', '01473000', '01474000', '01477000', '01479000', '01480000', '01480500', '01484100', '01485500', '01486000', '01487000', '01491000', '01495000', '01500000', '01509000', '01510000', '01516500', '01518862', '01523500', '01524500', '01532000', '01534000', '01536000', '01538000', '01539000', '01541000', '01541200', '01541500', '01542810', '01543000', '01543500', '01544000', '01544500', '01545000', '01545600', '01546500', '01547500', '01547700', '01547950', '01548500', '01549500', '01550000', '01552000', '01552500', '01555000', '01555500', '01556000', '01557500', '01558000', '01560000', '01564500', '01567500', '01568000', '01571500', '01573000', '01574500', '01576500', '01580000', '01582000', '01583500', '01586000', '01586610', '01591400', '01594950', '01595200', '01596500', '01599000', '01601500', '01604500', '01605500', '01606500', '01607500', '01613050', '01616500', '01619500', '01620500', '01625000', '01626000', '01632000', '01632900', '01634500', '01635500', '01637500', '01638480', '01639000', '01639500', '01643500', '01644000', '01645000', '01646000', '01648000', '01649500', '01651000', '01653600', '01658500', '01664000', '01666500', '01667500', '01669000', '01669520', '01673800', '01674000', '02011400', '02011460', '02013000', '02014000', '02015700', '02016000', '02017500', '02018000', '02021500', '02027000', '02027500', '02028500', '02038850', '02039000', '02039500', '02041000', '02042500', '02044500', '02046000', '02051000', '02051500', '02053200', '02053500', '02053800', '02054500', '02055100', '02056900', '02058400', '02059500', '02064000', '02065500', '02069700', '02070000', '02073000', '02074500', '02077200', '02081500', '02082950', '02085500', '02088000', '02090380', '02091000', '02092500', '02094500', '02096846', '02100500', '02102908', '02108000', '02111180', '02111500', '02112120', '02112360', '02113000', '02113850', '02114450', '02118000', '02118500', '02125000', '02128000', '02133500', '02137727', '02138500', '02140991', '02143000', '02143040', '02143500', '02146600', '02149000', '02152100', '02169570', '02177000', '02178400', '02193340', '02196000', '02198100', '02202600', '02212600', '02213500', '02215100', '02216180', '02221525', '02231000', '02231342', '02235200', '02245500', '02246000', '02296500', '02297155', '02297310', '02298123', '02298608', '02299950', '02300700', '02310947', '02312200', '02314500', '02315500', '02324400', '02327100', '02331600', '02333500', '02335700', '02336300', '02337000', '02341800', '02342500', '02342933', '02344500', '02344700', '02349900', '02350900', '02361000', '02363000', '02369800', '02371500', '02372250', '02374500', '02380500', '02381600', '02384540', '02385800', '02395120', '02398000', '02399200', '02401390', '02408540', '02415000', '02422500', '02427250', '02430085', '02430615', '02450000', '02450250', '02464000', '02464146', '02464360', '02465493', '02469800', '02471001', '02472000', '02472500', '02473500', '02475500', '02476500', '02476600', '02479130', '02479155', '02479300', '02479560', '02481000', '02481510', '02484000', '03010655', '03011800', '03014500', '03015500', '03020500', '03021350', '03026500', '03028000', '03034000', '03034500', '03038000', '03042000', '03042280', '03042500', '03045000', '03049000', '03049800', '03050000', '03053500', '03061500', '03066000', '03069500', '03070500', '03072000', '03074500', '03076500', '03076600', '03078000', '03079000', '03080000', '03085500', '03091500', '03093000', '03102850', '03106000', '03108000', '03110000', '03111500', '03112000', '03117500', '03118000', '03118500', '03136500', '03140000', '03145000', '03157000', '03159540', '03161000', '03164000', '03165000', '03167000', '03170000', '03173000', '03180500', '03182500', '03186500', '03187500', '03202400', '03207800', '03208500', '03208950', '03209000', '03213700', '03217000', '03220000', '03228500', '03232500', '03237280', '03237500', '03238500', '03240000', '03241500', '03262000', '03264000', '03272000', '03275600', '03280700', '03281100', '03281500', '03283500', '03285000', '03291780', '03294000', '03300400', '03302800', '03303300', '03310300', '03320500', '03324000', '03325500', '03329700', '03330500', '03333450', '03334500', '03338780', '03340800', '03342500', '03343400', '03346000', '03347000', '03350700', '03351500', '03353500', '03357350', '03358000', '03361650', '03361850', '03362000', '03363500', '03364500', '03366500', '03368000', '03369500', '03375500', '03378000', '03378550', '03378635', '03382100', '03384450', '03438000', '03439000', '03446000', '03450000', '03451000', '03455500', '03456500', '03459500', '03460000', '03463300', '03471500', '03473000', '03479000', '03488000', '03491000', '03498500', '03500000', '03500240', '03504000', '03512000', '03531500', '03550000', '03574500', '03592718', '03604000', '03612000', '04015330', '04024430', '04027000', '04027500', '04036000', '04037500', '04040500', '04043050', '04045500', '04056500', '04057510', '04057800', '04059500', '04063700', '04074950', '04087120', '04087220', '04093000', '04094000', '04096405', '04099510', '04101800', '04102700', '04105000', '04105700', '04108600', '04108800', '04109000', '04112500', '04115265', '04117500', '04121300', '04122200', '04122500', '04124000', '04127918', '04127997', '04128990', '04142000', '04155000', '04161540', '04161580', '04164500', '04166500', '04168000', '04180000', '04185000', '04186500', '04189000', '04196800', '04197100', '04197170', '04201500', '04202000', '04207200', '04213000', '04213075', '04214500', '04215500', '04216418', '04221000', '04224775', '04230500', '04233000', '04234000', '04239000', '04243500', '04252500', '04256000', '04258000', '04262500', '04280000', '04282000', '04285500', '04287000', '04288000', '04289000', '04292000', '04296000', '04296500', '05017500', '05056000', '05057000', '05057200', '05062500', '05078230', '05087500', '05101000', '05120500', '05123400', '05129115', '05130500', '05131500', '05291000', '05315000', '05362000', '05370000', '05393500', '05394500', '05397500', '05399500', '05406500', '05408000', '05412500', '05413500', '05414000', '05414820', '05419000', '05429500', '05431486', '05444000', '05451900', '05452200', '05454000', '05454300', '05458000', '05463000', '05466000', '05466500', '05470500', '05481950', '05487980', '05488200', '05489000', '05495000', '05495500', '05501000', '05503800', '05506800', '05507600', '05508805', '05512500', '05514500', '05516500', '05522500', '05525500', '05527800', '05529000', '05536000', '05536190', '05536275', '05536290', '05539000', '05540095', '05540500', '05543830', '05548280', '05551200', '05551700', '05556500', '05568800', '05577500', '05579500', '05580000', '05584500', '05585000', '05587900', '05591550', '05592050', '05592575', '05593575', '05593900', '05594450', '05595730', '06037500', '06043500', '06061500', '06154400', '06154410', '06169500', '06188000', '06191500', '06211000', '06211500', '06216000', '06221400', '06224000', '06278300', '06280300', '06289000', '06291500', '06298000', '06309200', '06311000', '06332515', '06339100', '06339500', '06342260', '06342450', '06344600', '06350000', '06352000', '06353000', '06354000', '06360500', '06402000', '06402500', '06404000', '06406000', '06408700', '06409000', '06410500', '06422500', '06431500', '06440200', '06441500', '06447000', '06447500', '06449000', '06450500', '06452000', '06453600', '06464500', '06468170', '06468250', '06470800', '06477500', '06478540', '06478690', '06479215', '06479438', '06600100', '06601000', '06614800', '06622700', '06623800', '06625000', '06632400', '06647500', '06712000', '06746095', '06784000', '06800000', '06803000', '06803510', '06803520', '06803530', '06804000', '06806500', '06814000', '06847900', '06853800', '06871500', '06876700', '06878000', '06879650', '06884200', '06885500', '06888500', '06889200', '06889500', '06892000', '06893500', '06894000', '06903400', '06903700', '06906800', '06910800', '06911900', '06912500', '06915000', '06917000', '06918440', '06918460', '06918740', '06919500', '06921070', '06921200', '06932000', '06934000', '07014500', '07017200', '07050700', '07056000', '07057500', '07060710', '07066000', '07067000', '07068000', '07071500', '07083000', '07103700', '07126200', '07142300', '07145700', '07148400', '07149000', '07151500', '07167500', '07169800', '07179500', '07179795', '07180500', '07184000', '07191220', '07195800', '07196900', '07197000', '07203000', '07207000', '07207500', '07208500', '07216500', '07218000', '07226500', '07230000', '07247500', '07249400', '07261000', '07263295', '07264000', '07274000', '07283000', '07290650', '07291000', '07292500', '07295000', '07299670', '07301410', '07301500', '07315200', '07315700', '07325800', '07326000', '07335700', '07337900', '07340300', '07343000', '07346045', '07352000', '07359610', '07361500', '07362100', '07362587', '07373000', '07375000', '07376000', '07376500', '07383500', '08013000', '08014500', '08017200', '08017300', '08019500', '08023080', '08025500', '08028000', '08029500', '08041700', '08050800', '08061540', '08063100', '08063800', '08064700', '08065200', '08066170', '08066200', '08066300', '08069000', '08070000', '08070200', '08070500', '08074500', '08076000', '08079600', '08082700', '08083100', '08086212', '08086290', '08101000', '08103900', '08104900', '08109700', '08109800', '08110100', '08150800', '08155200', '08158600', '08158700', '08158810', '08164000', '08164300', '08164600', '08164800', '08165300', '08171000', '08171300', '08172400', '08175000', '08176900', '08178800', '08178880', '08189500', '08189700', '08190000', '08190500', '08194200', '08195000', '08196000', '08198000', '08198500', '08200000', '08202700', '08255500', '08265000', '08267500', '08269000', '08271000', '08275500', '08276300', '08279000', '08291000', '08324000', '08377900', '08378500', '08380500', '08387000', '08401900', '08405500', '09034900', '09035800', '09035900', '09037500', '09047500', '09047700', '09050100', '09050700', '09059500', '09063000', '09065500', '09066000', '09066200', '09066300', '09073400', '09080400', '09081600', '09107000', '09112500', '09124500', '09146200', '09172500', '09210500', '09223000', '09255000', '09277500', '09289500', '09292500', '09306242', '09310500', '09312600', '09326500', '09330500', '09337500', '09342500', '09352900', '09372000', '09378170', '09378630', '09386900', '09390500', '09404450', '09409880', '09430500', '09430600', '09442680', '09447800', '09480000', '09484000', '09484600', '09492400', '09494000', '09497800', '09497980', '09504500', '09505200', '09505350', '09505800', '09507980', '09508300', '09510200', '09512280', '09513780', '09513860', '10011500', '10023000', '10032000', '10109000', '10128500', '10131000', '10134500', '10137500', '10166430', '10172700', '10172800', '10173450', '10174500', '10194200', '10205030', '10234500', '10242000', '10244950', '10249300', '10258000', '10258500', '10259000', '10259200', '10263500', '10293000', '10296000', '10296500', '10308200', '10309000', '10310000', '10310500', '10316500', '10329500', '10336645', '10336660', '10336740', '10343500', '10344400', '10344500', '10348850', '10349300', '10352500', '10396000', '11012500', '11015000', '11022480', '11025500', '11028500', '11042400', '11065000', '11073360', '11077500', '11098000', '11109600', '11118500', '11123000', '11123500', '11124500', '11132500', '11141280', '11143000', '11143200', '11143250', '11147500', '11148900', '11149900', '11151300', '11152000', '11156500', '11160000', '11160500', '11162500', '11176400', '11176500', '11180500', '11180960', '11200800', '11224500', '11230500', '11237500', '11253310', '11264500', '11266500', '11274500', '11274630', '11284400', '11299600', '11345500', '11376550', '11379500', '11381500', '11383500', '11390000', '11402000', '11413000', '11424000', '11427000', '11451100', '11458000', '11461000', '11468000', '11468500', '11473900', '11475560', '11476600', '11478500', '11480390', '11481200', '11482500', '11521500', '11522500', '11523200', '11528700', '11532500', '12010000', '12013500', '12020000', '12025000', '12025700', '12035000', '12036000', '12040500', '12041200', '12043000', '12048000', '12054000', '12056500', '12073500', '12082500', '12086500', '12092000', '12093500', '12095000', '12112600', '12114500', '12115000', '12115500', '12117000', '12141300', '12142000', '12143600', '12144000', '12144500', '12145500', '12147500', '12147600', '12148500', '12167000', '12175500', '12178100', '12186000', '12189500', '12332000', '12335500', '12342500', '12358500', '12374250', '12375900', '12377150', '12381400', '12383500', '12388400', '12390700', '12411000', '12414500', '12414900', '12447390', '12451000', '12452800', '12488500', '12502500', '13011500', '13011900', '13018300', '13023000', '13052200', '13075000', '13083000', '13120000', '13161500', '13174500', '13200000', '13235000', '13239000', '13240000', '13310700', '13313000', '13331500', '13337000', '13338500', '13340000', '13340600', '13345000', '14020000', '14034500', '14092750', '14096850', '14123500', '14137000', '14138800', '14138870', '14138900', '14139800', '14141500', '14145500', '14151000', '14154500', '14155500', '14158500', '14158790', '14159500', '14162200', '14165000', '14166500', '14170000', '14178000', '14182500', '14185000', '14185900', '14187000', '14190500', '14203500', '14216500', '14222500', '14226500', '14236200', '14301000', '14301500', '14303200', '14305500', '14306340', '14306500', '14308990', '14309000', '14309500', '14316700', '14325000', '14337500', '14338000', '14362000', '14362250', '14377100', '14400000']
gage_ids = ALL_CAMELS_GAGE_IDS if RUN_FULL_GAGE_SET else ALL_CAMELS_GAGE_IDS[:DEMONSTRATION_GAGE_COUNT]
print(f"Selected {len(gage_ids)} of {len(ALL_CAMELS_GAGE_IDS)} embedded CAMELS gages")
print(gage_ids)


## 3. Dask client and HyTEST CONUS404 access

The notebook opens the public HyTEST catalog, selects `conus404-daily-osn`, parses the model projection, validates `ACRUNSF` and `ACQRF`, and limits the lazy dataset to the analysis period before computation.


In [ ]:
client = Client(n_workers=DASK_WORKERS, threads_per_worker=1)
hytest_catalog = intake.open_catalog(HYTEST_CATALOG_URL)
conus404_catalog = hytest_catalog["conus404-catalog"]
conus404 = conus404_catalog[CONUS404_DATASET].to_dask().metpy.parse_cf()
required_variables = {"ACRUNSF", "ACQRF"}
missing_variables = sorted(required_variables.difference(conus404.data_vars))
if missing_variables:
    raise KeyError(f"CONUS404 is missing required runoff variables: {missing_variables}")
conus404_daily = conus404[["ACRUNSF", "ACQRF"]].sel(time=slice(START_DATE, END_DATE))
print(conus404_daily)


## 4. Annual CONUS404 total runoff

Total runoff is the sum of accumulated surface runoff and accumulated subsurface runoff:

$$
Q_{CONUS404}=\mathrm{ACRUNSF}+\mathrm{ACQRF}.
$$

Daily depths are summed from October through September. The right-hand label assigns each result to the ending water year. `min_count=1` preserves a fully missing period as nodata rather than converting it to zero.


In [ ]:
runoff_daily = (conus404_daily["ACRUNSF"] + conus404_daily["ACQRF"]).rename("Runoff")
runoff_daily.attrs.update({"long_name": "CONUS404 total runoff", "units": "mm", "construction": "ACRUNSF + ACQRF"})
runoff_water_year = runoff_daily.resample(time="YS-OCT", closed="left", label="right").sum(min_count=1)
conus404_crs = conus404.crs.metpy.cartopy_crs
runoff_water_year = runoff_water_year.rio.set_spatial_dims(x_dim="x", y_dim="y").rio.write_crs(conus404_crs)
runoff_water_year


## 5. NLDI basin polygons and gage locations

NLDI returns the contributing polygon and the `nwissite` point for every selected USGS identifier. Basin areas are calculated in the CONUS Albers equal-area projection. Failed service requests remain listed instead of being silently removed.


In [ ]:
nldi = NLDI()
basin_records = []
gage_records = []
failed_nldi_requests = {}
for gage_id in gage_ids:
    try:
        basin = nldi.get_basins(gage_id).copy()
        basin["Gage_ID"] = gage_id
        basin_records.append(basin[["Gage_ID", "geometry"]])
        gage = nldi.getfeature_byid("nwissite", f"USGS-{gage_id}").copy()
        gage["Gage_ID"] = gage_id
        gage_records.append(gage[["Gage_ID", "geometry"]])
    except Exception as error:
        failed_nldi_requests[gage_id] = str(error)

if not basin_records:
    raise RuntimeError(f"NLDI did not return any basin polygons: {failed_nldi_requests}")
basins = gpd.GeoDataFrame(pd.concat(basin_records, ignore_index=True), crs=basin_records[0].crs)
basins = basins.dissolve(by="Gage_ID", as_index=False)
basins_equal_area = basins.to_crs("EPSG:5070")
basins_equal_area["Area_sqkm"] = basins_equal_area.geometry.area / 1_000_000.0
if gage_records:
    gage_points = gpd.GeoDataFrame(pd.concat(gage_records, ignore_index=True), crs=gage_records[0].crs)
    gage_points = gage_points.drop_duplicates(subset="Gage_ID")
else:
    gage_points = basins_equal_area.copy()
    gage_points.geometry = gage_points.geometry.representative_point()
    gage_points = gage_points.to_crs("EPSG:4326")
basins.to_file(OUTPUT_DIRECTORY / "nldi_camels_basins.gpkg", layer="basins", driver="GPKG")
gage_points.to_file(OUTPUT_DIRECTORY / "nldi_gage_points.gpkg", layer="gages", driver="GPKG")
print(basins_equal_area[["Gage_ID", "Area_sqkm"]])
print(failed_nldi_requests)


## 6. Computational bounding box and CONUS404 grid-cell polygons

Basin polygons are projected to the CONUS404 grid. Their combined bounding box limits the raster before grid polygons are constructed from the x- and y-coordinate bounds.


In [ ]:
basins_grid_crs = basins.to_crs(runoff_water_year.rio.crs)
minimum_x, minimum_y, maximum_x, maximum_y = basins_grid_crs.total_bounds
x_resolution = float(np.abs(np.diff(runoff_water_year.x.values)).mean())
y_resolution = float(np.abs(np.diff(runoff_water_year.y.values)).mean())

def ordered_slice(coordinate, lower, upper, buffer):
    return slice(lower - buffer, upper + buffer) if coordinate[0] < coordinate[-1] else slice(upper + buffer, lower - buffer)

runoff_subset = runoff_water_year.sel(
    x=ordered_slice(runoff_water_year.x.values, minimum_x, maximum_x, x_resolution),
    y=ordered_slice(runoff_water_year.y.values, minimum_y, maximum_y, y_resolution),
)
grid = xr.Dataset(coords={"x": runoff_subset.x, "y": runoff_subset.y}).cf.add_bounds(["x", "y"])
points = grid.stack(point=("y", "x"))

def bounds_to_polygon(x_bounds, y_bounds):
    return Polygon(((x_bounds[0], y_bounds[0]), (x_bounds[0], y_bounds[1]), (x_bounds[1], y_bounds[1]), (x_bounds[1], y_bounds[0])))

boxes = xr.apply_ufunc(
    bounds_to_polygon,
    points.x_bounds,
    points.y_bounds,
    input_core_dims=[["bounds"], ["bounds"]],
    output_dtypes=[np.dtype("O")],
    vectorize=True,
)
grid_cells = gpd.GeoDataFrame(
    {"y": boxes["y"].values, "x": boxes["x"].values},
    geometry=boxes.values,
    crs=runoff_subset.rio.crs,
)
print(f"Raster subset: {runoff_subset.sizes}")
print(f"Grid polygons: {len(grid_cells):,}")


## 7. Basin–grid intersections and area fractions

Grid cells and basins are projected to EPSG:5070 and intersected. Each intersection area is divided by the total intersected area of its basin. Consequently, the cell fractions sum to one for every basin and represent an area-weighted basin mean.


In [ ]:
grid_equal_area = grid_cells.to_crs("EPSG:5070")
basin_overlay = basins_equal_area[["Gage_ID", "geometry"]]
overlay = gpd.overlay(grid_equal_area, basin_overlay, how="intersection", keep_geom_type=False)
overlay["intersection_area_m2"] = overlay.geometry.area
overlay = overlay.loc[overlay["intersection_area_m2"] > 0].copy()
basin_intersection_area = overlay.groupby("Gage_ID")["intersection_area_m2"].transform("sum")
overlay["weight"] = overlay["intersection_area_m2"] / basin_intersection_area
weight_sums = overlay.groupby("Gage_ID")["weight"].sum()
if not np.allclose(weight_sums.to_numpy(), 1.0):
    raise ValueError("One or more basin weight sets do not sum to one.")
weight_rows = overlay.groupby(["y", "x", "Gage_ID"], as_index=False)["weight"].sum()
print(weight_sums)


## 8. Sparse basin-weight matrix

The overlay table is converted to a sparse three-dimensional matrix indexed by raster y, raster x, and gage. This preserves the source notebook's sparse-matrix aggregation without constructing a dense grid-by-basin array.


In [ ]:
y_values = runoff_subset.y.values
x_values = runoff_subset.x.values
active_gage_ids = sorted(weight_rows["Gage_ID"].unique())
y_lookup = {float(value): index for index, value in enumerate(y_values)}
x_lookup = {float(value): index for index, value in enumerate(x_values)}
gage_lookup = {value: index for index, value in enumerate(active_gage_ids)}
sparse_coordinates = np.vstack(
    (
        weight_rows["y"].map(y_lookup).to_numpy(),
        weight_rows["x"].map(x_lookup).to_numpy(),
        weight_rows["Gage_ID"].map(gage_lookup).to_numpy(),
    )
)
sparse_data = sparse.COO(
    sparse_coordinates,
    weight_rows["weight"].to_numpy(dtype="float64"),
    shape=(len(y_values), len(x_values), len(active_gage_ids)),
)
basin_weights = xr.DataArray(
    sparse_data,
    dims=("y", "x", "Gage_ID"),
    coords={"y": y_values, "x": x_values, "Gage_ID": active_gage_ids},
    name="basin_area_fraction",
)
basin_weights


## 9. Area-weighted CONUS404 runoff by basin

Sparse matrix multiplication combines every annual runoff grid with the basin fractions. The result is one annual CONUS404 runoff depth for each NLDI basin.


In [ ]:
def apply_sparse_basin_weights(weights, data):
    data_array = np.asarray(data)
    original_dimension = data_array.ndim
    if original_dimension == 2:
        data_array = data_array[np.newaxis, ...]
    sparse_runoff = sparse.COO.from_numpy(data_array)
    time_count, y_count, x_count = sparse_runoff.shape
    runoff_matrix = sparse_runoff.reshape((time_count, y_count * x_count))
    weight_matrix = weights.reshape((y_count * x_count, weights.shape[2]))
    result = sparse.matmul(runoff_matrix, weight_matrix).todense()
    return result[0] if original_dimension == 2 else result

with dask.config.set({"array.slicing.split_large_chunks": False}):
    basin_model_runoff = xr.apply_ufunc(
        apply_sparse_basin_weights,
        basin_weights,
        runoff_subset,
        input_core_dims=[["y", "x", "Gage_ID"], ["y", "x"]],
        output_core_dims=[["Gage_ID"]],
        dask="parallelized",
        output_dtypes=["float64"],
        dask_gufunc_kwargs={"output_sizes": {"Gage_ID": len(active_gage_ids)}, "allow_rechunk": True},
    ).compute()
basin_model_runoff.name = "Runoff_Squaw"
basin_model_runoff


## 10. Spatial and temporal inspection

The figures display the first annual CONUS404 runoff grid with NLDI basin boundaries and the complete basin-mean runoff series. They replace the source notebook's temporary shapefile reload and browser-dependent interactive maps while retaining the same spatial and temporal checks.


In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(16, 6))
runoff_subset.isel(time=0).plot(ax=axes[0], cmap="turbo", robust=True)
basins_grid_crs.boundary.plot(ax=axes[0], color="black", linewidth=0.8)
axes[0].set_title(f"CONUS404 runoff and NLDI basins: WY {pd.Timestamp(runoff_subset.time.values[0]).year}")
for gage_id in basin_model_runoff.Gage_ID.values:
    axes[1].plot(pd.to_datetime(basin_model_runoff.time.values), basin_model_runoff.sel(Gage_ID=gage_id), label=str(gage_id))
axes[1].set_xlabel("Water year")
axes[1].set_ylabel("CONUS404 runoff (mm)")
axes[1].legend(fontsize=7, ncol=2)
axes[1].grid(alpha=0.25)
figure.tight_layout()
figure.savefig(OUTPUT_DIRECTORY / "conus404_runoff_spatial_and_temporal.png", dpi=300, bbox_inches="tight")
plt.show()


## 11. Daily USGS discharge from NWIS

The NWIS daily-values query retrieves statistic `00003` for parameter `00060`, representing daily mean discharge. The source values are converted from cubic feet per second to cubic meters per second. Each unsuccessful gage request is recorded.


In [ ]:
daily_discharge = {}
failed_nwis_requests = {}
for gage_id in active_gage_ids:
    try:
        response = hf.NWIS(gage_id, "dv", start_date=START_DATE, end_date=END_DATE)
        frame = response.df()
        frame.index = pd.to_datetime(frame.index, errors="coerce").tz_localize(None)
        expected_column = f"USGS:{gage_id}:00060:00003"
        candidates = [column for column in frame.columns if str(column).startswith(expected_column)]
        if not candidates:
            raise KeyError(f"NWIS did not return {expected_column}")
        daily_discharge[gage_id] = pd.to_numeric(frame[candidates[0]], errors="coerce") * 0.028316846592
    except Exception as error:
        failed_nwis_requests[gage_id] = str(error)
if not daily_discharge:
    raise RuntimeError(f"NWIS did not return discharge for any gage: {failed_nwis_requests}")
daily_discharge_m3s = pd.DataFrame(daily_discharge).sort_index()
daily_discharge_m3s.to_csv(OUTPUT_DIRECTORY / "nwis_daily_discharge_m3s.csv", index_label="date")
print(daily_discharge_m3s.notna().sum())
print(failed_nwis_requests)


## 12. USGS discharge converted to annual runoff depth

Daily mean discharge is converted to daily volume and divided by NLDI basin area:

$$
Q_{USGS,\,WY}(\mathrm{mm})=
\frac{\sum_d q_d(\mathrm{m^3\,s^{-1}})\,86400}{A_b(\mathrm{m^2})}\,1000.
$$

An annual value is retained only when its daily-data coverage meets the configured fraction. This avoids scaling an incomplete year as though it were complete.


In [ ]:
daily_volume_m3 = daily_discharge_m3s * 86400.0
annual_volume_m3 = daily_volume_m3.resample("YS-OCT", closed="left", label="right").sum(min_count=1)
annual_counts = daily_discharge_m3s.resample("YS-OCT", closed="left", label="right").count()
calendar = pd.Series(1.0, index=pd.date_range(START_DATE, END_DATE, freq="D"))
expected_counts = calendar.resample("YS-OCT", closed="left", label="right").sum()
coverage = annual_counts.div(expected_counts, axis=0)
area_m2 = basins_equal_area.set_index("Gage_ID").geometry.area
observed_runoff_mm = annual_volume_m3.divide(area_m2, axis=1) * 1000.0
observed_runoff_mm = observed_runoff_mm.where(coverage >= MINIMUM_DAILY_COVERAGE)
coverage.to_csv(OUTPUT_DIRECTORY / "nwis_daily_coverage_by_water_year.csv", index_label="Year")
observed_runoff_mm


## 12a. Streamflow completeness diagnostics

The coverage diagnostics preserve the source notebook's missing-data assessment. They report daily completeness for every gage and the mean annual completeness across the selected basin set.


In [ ]:
gage_completeness = pd.DataFrame(
    {
        "Gage_ID": daily_discharge_m3s.columns,
        "Available_daily_values": daily_discharge_m3s.notna().sum().to_numpy(),
        "Expected_daily_values": len(daily_discharge_m3s),
        "Available_percent": 100.0 * daily_discharge_m3s.notna().mean().to_numpy(),
        "Missing_percent": 100.0 * daily_discharge_m3s.isna().mean().to_numpy(),
    }
)
gage_completeness.to_csv(OUTPUT_DIRECTORY / "nwis_daily_completeness_by_gage.csv", index=False)
mean_annual_completeness = 100.0 * coverage.mean(axis=1)
figure, axis = plt.subplots(figsize=(11, 5))
axis.plot(mean_annual_completeness.index.year, mean_annual_completeness, marker="o", color="#2166ac")
axis.set_xlabel("Ending water year")
axis.set_ylabel("Mean available daily values (%)")
axis.set_title("NWIS daily-discharge completeness across selected gages")
axis.set_ylim(0, 100)
axis.grid(alpha=0.25)
figure.tight_layout()
figure.savefig(OUTPUT_DIRECTORY / "nwis_annual_data_completeness.png", dpi=300, bbox_inches="tight")
plt.show()
gage_completeness


## 13. Alignment of CONUS404 and USGS annual runoff

Both products are converted to a long table and joined by gage identifier and ending water year. The exported column names remain compatible with the manuscript runoff-accuracy notebook.


In [ ]:
modeled = basin_model_runoff.to_dataframe().reset_index()
modeled["Year"] = pd.to_datetime(modeled["time"]).dt.year
modeled = modeled.rename(columns={"Gage_ID": "Gage_ID"})[["Gage_ID", "Year", "Runoff_Squaw"]]
modeled["Gage_ID"] = modeled["Gage_ID"].astype(str).str.zfill(8)

observed = observed_runoff_mm.rename_axis(index="time", columns="Gage_ID").stack(dropna=False).rename("Runoff_Annual").reset_index()
observed["Year"] = pd.to_datetime(observed["time"]).dt.year
observed["Gage_ID"] = observed["Gage_ID"].astype(str).str.zfill(8)
observed = observed[["Gage_ID", "Year", "Runoff_Annual"]]

aligned_data = observed.merge(modeled, on=["Gage_ID", "Year"], how="inner")
aligned_data = aligned_data.loc[np.isfinite(aligned_data["Runoff_Annual"]) & np.isfinite(aligned_data["Runoff_Squaw"])].copy()
aligned_data.to_csv(OUTPUT_DIRECTORY / "aligned_annual_runoff.csv", index=False)
print(aligned_data)


## 14. Basin accuracy statistics

For each basin, the notebook calculates mean bias error, MAE, RMSE, PBIAS, and nMAE from the paired annual depths. The paper definition normalizes MAE by the multiannual modeled CONUS404 runoff:

$$
\mathrm{PBIAS}(\%)=100\frac{\sum_t(S_t-O_t)}{\sum_tO_t},\qquad
\mathrm{nMAE}(\%)=100\frac{\mathrm{MAE}}{\overline{S}}.
$$


In [ ]:
metric_rows = []
for gage_id, group in aligned_data.groupby("Gage_ID", sort=True):
    observed_values = group["Runoff_Annual"].to_numpy(dtype="float64")
    modeled_values = group["Runoff_Squaw"].to_numpy(dtype="float64")
    difference = modeled_values - observed_values
    mae = np.mean(np.abs(difference))
    observed_sum = np.sum(observed_values)
    modeled_mean = np.mean(modeled_values)
    metric_rows.append(
        {
            "Gage_ID": gage_id,
            "N": len(group),
            "Bias_MBE_mm": np.mean(difference),
            "MAE_mm": mae,
            "RMSE_mm": np.sqrt(np.mean(difference ** 2)),
            "PBIAS_percent": 100.0 * np.sum(difference) / observed_sum if observed_sum > 0 else np.nan,
            "CONUS404_multiannual_mean_mm": modeled_mean,
            "nMAE_fraction": mae / modeled_mean if modeled_mean > 0 else np.nan,
            "nMAE_percent": 100.0 * mae / modeled_mean if modeled_mean > 0 else np.nan,
        }
    )
basin_metrics = pd.DataFrame(metric_rows).merge(
    basins_equal_area[["Gage_ID", "Area_sqkm"]], on="Gage_ID", how="left"
)
basin_metrics.to_csv(OUTPUT_DIRECTORY / "basin_accuracy_metrics.csv", index=False)
basin_metrics


## 15. Individual regression scatter plots and coefficient table

Every basin with the minimum number of paired years receives a scatter plot, fitted linear-regression line, 1:1 line, slope, intercept, and coefficient of determination. The figures compare USGS runoff depth on the x-axis with CONUS404 runoff depth on the y-axis.


In [ ]:
regression_rows = []
for gage_id, group in aligned_data.groupby("Gage_ID", sort=True):
    group = group.dropna(subset=["Runoff_Annual", "Runoff_Squaw"])
    if len(group) < MINIMUM_REGRESSION_PAIRS:
        continue
    x_values = group["Runoff_Annual"].to_numpy(dtype="float64").reshape(-1, 1)
    y_values = group["Runoff_Squaw"].to_numpy(dtype="float64")
    model = LinearRegression().fit(x_values, y_values)
    predictions = model.predict(x_values)
    r_squared = model.score(x_values, y_values)
    regression_rows.append(
        {"Gage_ID": gage_id, "N": len(group), "R_Squared": r_squared, "Slope": model.coef_[0], "Intercept": model.intercept_}
    )
    lower = min(float(x_values.min()), float(y_values.min()))
    upper = max(float(x_values.max()), float(y_values.max()))
    figure, axis = plt.subplots(figsize=(7, 6))
    axis.scatter(x_values[:, 0], y_values, color="black", alpha=0.75)
    order = np.argsort(x_values[:, 0])
    axis.plot(x_values[order, 0], predictions[order], color="#2166ac", label=f"Fit: y={model.coef_[0]:.2f}x+{model.intercept_:.2f}")
    axis.plot([lower, upper], [lower, upper], color="gray", linestyle="--", label="1:1 line")
    axis.set_xlabel("USGS runoff (mm)")
    axis.set_ylabel("CONUS404 runoff (mm)")
    axis.set_title(f"Gage {gage_id}: R²={r_squared:.2f}, N={len(group)}")
    axis.legend()
    axis.grid(alpha=0.25)
    figure.tight_layout()
    figure.savefig(SCATTER_DIRECTORY / f"{gage_id}_scatter_plot.png", dpi=300, bbox_inches="tight")
    plt.close(figure)
regression_results = pd.DataFrame(regression_rows)
regression_results.to_csv(OUTPUT_DIRECTORY / "basin_regression_statistics.csv", index=False)
regression_results


## 16. Filtered multi-basin scatter grid

The poster-style grid reproduces the source filtering logic. It retains basins above the configured R² threshold and paired-year threshold and displays each fitted relationship with its 1:1 line.


In [ ]:
selected_regressions = regression_results.loc[
    (regression_results["R_Squared"] > SCATTER_GRID_R2_MINIMUM)
    & (regression_results["N"] > SCATTER_GRID_PAIR_MINIMUM)
] if not regression_results.empty else regression_results
column_count = 4
panel_count = max(1, len(selected_regressions))
row_count = math.ceil(panel_count / column_count)
figure, axes = plt.subplots(row_count, column_count, figsize=(5 * column_count, 4 * row_count), squeeze=False)
for axis in axes.flat:
    axis.axis("off")
for axis, row in zip(axes.flat, selected_regressions.itertuples(index=False)):
    group = aligned_data.loc[aligned_data["Gage_ID"] == row.Gage_ID]
    x_values = group["Runoff_Annual"].to_numpy(dtype="float64")
    y_values = group["Runoff_Squaw"].to_numpy(dtype="float64")
    predictions = row.Slope * x_values + row.Intercept
    order = np.argsort(x_values)
    lower = min(float(x_values.min()), float(y_values.min()))
    upper = max(float(x_values.max()), float(y_values.max()))
    axis.axis("on")
    axis.scatter(x_values, y_values, color="black", alpha=0.75, s=16)
    axis.plot(x_values[order], predictions[order], color="#2166ac")
    axis.plot([lower, upper], [lower, upper], color="gray", linestyle="--")
    axis.set_title(f"Gage {row.Gage_ID}\nR²={row.R_Squared:.2f}, N={row.N}")
    axis.set_xlabel("USGS runoff (mm)")
    axis.set_ylabel("CONUS404 runoff (mm)")
if selected_regressions.empty:
    axes[0, 0].axis("on")
    axes[0, 0].text(0.5, 0.5, "No basin met both filters", ha="center", va="center")
    axes[0, 0].set_xticks([])
    axes[0, 0].set_yticks([])
figure.tight_layout()
figure.savefig(OUTPUT_DIRECTORY / "filtered_basin_scatter_grid.png", dpi=600, bbox_inches="tight")
plt.show()


## 17. Basin area and error relationships

The basin-area analysis plots MAE against NLDI basin area on a logarithmic x-axis. It also reports the percentage of large basins below the configured nMAE threshold and searches log-spaced area classes for the largest proportion below the target nMAE.


In [ ]:
area_metrics = basin_metrics.dropna(subset=["Area_sqkm", "MAE_mm", "nMAE_percent"]).copy()
area_metrics = area_metrics.loc[area_metrics["Area_sqkm"] > 0]
area_metrics["MAE_class"] = pd.cut(
    area_metrics["MAE_mm"],
    bins=[-np.inf, 100.0, 200.0, np.inf],
    labels=["<100 mm", "100–200 mm", ">200 mm"],
)
colors = {"<100 mm": "#67a9cf", "100–200 mm": "#1c9099", ">200 mm": "#016c59"}
figure, axis = plt.subplots(figsize=(8, 6))
for label, subset in area_metrics.groupby("MAE_class", observed=True):
    axis.scatter(subset["Area_sqkm"], subset["MAE_mm"], color=colors[str(label)], label=str(label), alpha=0.85)
axis.set_xscale("log")
axis.set_xlabel("Basin area (km²)")
axis.set_ylabel("MAE (mm)")
axis.set_title("Runoff MAE versus NLDI basin area")
axis.legend(title="MAE class", frameon=False)
axis.grid(alpha=0.25, which="both")
figure.tight_layout()
figure.savefig(OUTPUT_DIRECTORY / "mae_vs_basin_area_logx.png", dpi=600, bbox_inches="tight")
plt.show()

area_threshold_km2 = 2500.0
nmae_threshold_percent = 25.0
large_basins = area_metrics.loc[area_metrics["Area_sqkm"] >= area_threshold_km2]
large_basin_percentage = 100.0 * (large_basins["nMAE_percent"] < nmae_threshold_percent).mean() if len(large_basins) else np.nan
print(f"Basins with area ≥ {area_threshold_km2:g} km²: {len(large_basins)}")
print(f"Percentage with nMAE < {nmae_threshold_percent:g}%: {large_basin_percentage:.2f}%")


In [ ]:
minimum_basins_per_bin = 3
number_of_area_bins = 10
area_bin_rows = []
if len(area_metrics) >= minimum_basins_per_bin and area_metrics["Area_sqkm"].nunique() > 1:
    edges = np.logspace(
        np.log10(area_metrics["Area_sqkm"].min()),
        np.log10(area_metrics["Area_sqkm"].max()),
        number_of_area_bins + 1,
    )
    for lower, upper in zip(edges[:-1], edges[1:]):
        subset = area_metrics.loc[(area_metrics["Area_sqkm"] >= lower) & (area_metrics["Area_sqkm"] < upper)]
        if len(subset) >= minimum_basins_per_bin:
            area_bin_rows.append(
                {
                    "Area_low_km2": lower,
                    "Area_high_km2": upper,
                    "N_basins": len(subset),
                    "Percent_nMAE_below_target": 100.0 * (subset["nMAE_percent"] < nmae_threshold_percent).mean(),
                }
            )
area_bin_results = pd.DataFrame(area_bin_rows)
if not area_bin_results.empty:
    area_bin_results = area_bin_results.sort_values(
        ["Percent_nMAE_below_target", "N_basins"], ascending=[False, False]
    ).reset_index(drop=True)
area_bin_results.to_csv(OUTPUT_DIRECTORY / "basin_area_error_classes.csv", index=False)
area_bin_results


## 18. Output inventory and client shutdown

The workflow writes the NLDI vectors, daily NWIS observations, coverage table, aligned annual runoff table, basin metrics, regression coefficients, individual scatter plots, filtered scatter grid, spatial and temporal inspection figure, and basin-area analysis below `results/runoff_hytest_nldi/`.


In [ ]:
output_inventory = pd.DataFrame(
    {
        "path": [str(path.relative_to(REPOSITORY_ROOT)) for path in sorted(OUTPUT_DIRECTORY.rglob("*")) if path.is_file()],
        "bytes": [path.stat().st_size for path in sorted(OUTPUT_DIRECTORY.rglob("*")) if path.is_file()],
    }
)
client.close()
output_inventory
